# **Laboratorio 5: Bolsa de Palabras** ⚙️🖼️

**Asignatura:** Visión por Ordenador I  
**Grado:** Ingeniería Matemática e Inteligencia Artificial  
**Curso:** 2026/2027  

### **Objetivo**
En esta parte de la práctica, aprenderá a implementar la Bolsa de Palabras.

## **Librerías**

In [1]:
! pip install tqdm scikit-learn pandas openpyxl

  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.3 MB 3.1 MB/s eta 0:00:03
   ------ --------------------------------- 1.3/8.3 MB 3.1 MB/s eta 0:00:03
   -------- ------------------------------- 1.8/8.3 MB 3.1 MB/s eta 0:00:03
   ------------ --------------------------- 2.6/8.3 MB 3.0 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.3 MB 3.0 MB/s eta 0:00:02
   ----------------- ---------------------- 3.7/8.3 MB 2.8 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.3 MB 2.8 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.3 MB 2.9 MB/s eta 0:00:02
   ------------------------- -------------- 5.2/8.3 MB 2.7 MB/s eta 0:00:02
   ----------------------------- ---------- 6


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\lloza\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


Solamente si lo anterior no funciona ejecutar esta opción

In [2]:
import sys
!{sys.executable} -m pip install tqdm scikit-learn pandas openpyxl

  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached tzdata-2026.6-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   -------- ------------------------------- 2.4/11.2 MB 19.1 MB/s eta 0:00:01
   ---------------------- ----------------- 6.3/11.2 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 18.6 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 17.4 MB/s  0:00:00
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ----------------- ---------------------- 5.0/11.4 MB 25.2 MB/s eta 0:00:01
   ---------------------------------------- 11.4/11.4 MB 28.5 MB/s  0:00:00
Using cache

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from bow import BoW
from dataset import Dataset
from image_classifier import ImageClassifier
import time
from tqdm import tqdm
import sys
import pickle
import os

# **Apartado A**: Bolsa de palabras

### **Tarea PREVIA:** Generamos la carpeta "../results/partA", junto con la subcarpeta "bow".

In [4]:
results_path = "../results/partA"
bow_path = "../results/partA/bow"
os.makedirs(results_path)
os.makedirs(bow_path)

### **Tarea A.1**: Carga de los datasets de entrenamiento y validación para la bolsa de palabras

Descargue el dataset desde el [`enlace en Moodle`](https://sifo.comillas.edu/pluginfile.php/5039369/mod_page/content/8/dataset.zip), descomprimalo en data/partA. 

Empleando el método `load()` de la clase [`Dataset`](.\dataset.py) que se le proporciona, cargue los datasets de entrenamiento y validación.

In [8]:
# TODO: Load the training and validation sets
training_set = Dataset.load("../data/partA/Dataset/dataset/training",".jpg")
validation_set = Dataset.load("../data/partA/Dataset/dataset/validation",".jpg")

### **Tarea A.2**: Extracción de los descriptores

Para poder crear la bolsa de palabras se va a comenzar obteniendo los descriptores que serán agrupados en las distintas palabras de nuestra bolsa. Para ello complete la carga de la imagen en la ruta `path` en escala de grises y obtenga sus descriptores con los métodos apropiados de cv2. Los descriptores obtenidos se añadirán a nuestra bolsa de palabras (`words`) para después ser agrupados por palabras que representen

In [9]:
feature_extractor = cv2.SIFT_create()

# Extract features
print("\nComputing SIFT descriptors...")
time.sleep(0.1)  # Prevents a race condition between tqdm and print statements.
descriptors = []
for path in tqdm(training_set, unit="image", file=sys.stdout):
    # TODO: Load the image from the path in grayscale
    image = cv2.imread(path,cv2.COLOR_BGR2GRAY)
    try:
        # TODO: Using the cv2 methods get the descriptors for the image
        _, descriptor = feature_extractor.detectAndCompute(image,None)
    except:
        print(f"WARN: Issue generating descriptor for image {path}")

    if descriptor is not None:
        descriptors.append(descriptor)


Computing SIFT descriptors...
100%|██████████| 2981/2981 [00:59<00:00, 49.89image/s]


### **Tarea A.3**: Creación del vocabulario

Con los descriptores extraídos se va a realizar su agrupamiento por palabras (K-Means) para obtener un vocabulario de palabras visuales. Agrupar los ~1.5 millones de descriptores del dataset es lento y consume mucha memoria, así que se toma una muestra aleatoria de 70 descriptores por imagen (unos 200 000 en total), que produce un vocabulario igual de útil. Añada la muestra a la bolsa (`words`) con su método `add()`. El vocabulario generado se guarda en un archivo para usarlo en las tareas siguientes.

In [10]:
# Build vocabulary
vocabulary_size = 100
iterations = 20
termination_criteria = (cv2.TERM_CRITERIA_MAX_ITER | cv2.TERM_CRITERIA_EPS, iterations, 1e-6)
words = cv2.BOWKMeansTrainer(vocabulary_size, termination_criteria)

# Random sample of (at most) 70 descriptors per image (~200 000 in total) to speed up K-means
rng = np.random.default_rng(0)
sample = np.vstack([d[rng.choice(len(d), min(len(d), 70), replace=False)] for d in descriptors])

# TODO: Add the sampled descriptors
words.add(sample)

time.sleep(0.1)  # Prevents a race condition between tqdm and print statements.
print("\nClustering descriptors into", vocabulary_size, "words using K-means...")
vocabulary = words.cluster()
filename=  f"{bow_path}/vocabulary"
with open(filename + ".pickle", "wb") as f:
    pickle.dump(["SIFT", vocabulary], f, pickle.HIGHEST_PROTOCOL)


Clustering descriptors into 100 words using K-means...


### **Tarea A.4**: Entrenamiento del clasificador

Una vez se tiene el vocabulario de palabras visuales, se va a entrenar al classificador. Para ello haga uso de la clase [BoW](./bow.py) y complete los métodos que se le indican

In [13]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary(filename)

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the training method
image_classifier.train(training_set)
classifier = f"{bow_path}/classifier"
image_classifier.save(classifier)



TRAINING CLASSIFIER

Extracting features...
100%|██████████| 2981/2981 [01:34<00:00, 31.64image/s]

Training SVM...


### **Tarea A.5**: Inferencia en dataset de entrenamiento
Con el modelo entrenado se procede a comprobar su desempeño con el dataset de entrenamiento. Complete los métodos indicados

In [ ]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary(f"{bow_path}/vocabulary")

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the loading method
image_classifier.load(classifier)
# TODO: Especify the args for the loading method
dataset_name = f"{bow_path}/training"
image_classifier.predict(training_set, dataset_name=dataset_name)



CLASSIFICATION RESULTS (../RESULTS/PARTA/BOW/TRAINING)

Confusion matrix

KNOWN/PREDICTED  Bedroom  Coast  Forest  Highway  Industrial  Inside city  Kitchen  Living room  Mountain  Office  Open country  Store  Street  Suburb  Tall building
Bedroom             31.0    3.0     2.0      1.0        10.0          5.0      2.0         25.0       5.0     2.0           3.0   10.0     8.0     4.0            5.0
Coast                3.0  192.0     7.0      7.0         2.0          0.0      0.0          1.0      15.0     0.0          19.0    0.0     3.0     6.0            5.0
Forest               0.0    0.0   209.0      0.0         0.0          0.0      0.0          0.0       9.0     0.0           9.0    0.0     1.0     0.0            0.0
Highway              0.0   34.0     0.0     69.0         8.0          5.0      0.0          2.0       4.0     0.0           7.0    2.0     9.0     8.0           12.0
Industrial           4.0    4.0     2.0      2.0       104.0         20.0      0.0         17.

(0.5961086883596108,
 array([[ 31.,   3.,   2.,   1.,  10.,   5.,   2.,  25.,   5.,   2.,   3.,  10.,   8.,   4.,   5.],
        [  3., 192.,   7.,   7.,   2.,   0.,   0.,   1.,  15.,   0.,  19.,   0.,   3.,   6.,   5.],
        [  0.,   0., 209.,   0.,   0.,   0.,   0.,   0.,   9.,   0.,   9.,   0.,   1.,   0.,   0.],
        [  0.,  34.,   0.,  69.,   8.,   5.,   0.,   2.,   4.,   0.,   7.,   2.,   9.,   8.,  12.],
        [  4.,   4.,   2.,   2., 104.,  20.,   0.,  17.,   9.,   3.,   2.,  16.,  11.,   8.,   9.],
        [  2.,   2.,   1.,   3.,  11., 140.,   1.,   6.,   0.,   1.,   2.,  14.,   9.,   2.,  14.],
        [ 10.,   0.,   0.,   0.,  15.,  22.,   9.,  29.,   0.,  10.,   0.,   5.,   2.,   2.,   5.],
        [  7.,   1.,   0.,   0.,  19.,  12.,   2., 111.,   1.,   5.,   1.,  13.,   5.,   8.,   4.],
        [  0.,  27.,  24.,   1.,   1.,   0.,   0.,   0., 186.,   0.,  28.,   0.,   4.,   1.,   1.],
        [  7.,   1.,   0.,   0.,  13.,  14.,   2.,  25.,   0.,  45.,   0.,   2.

### **Tarea A.6**: Inferencia en dataset de evaluación

A continuación se evalua el desempeño del modelo con el dataset de validación. Complete los métodos indicados

In [18]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary(f"{bow_path}/vocabulary")

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the loading method
image_classifier.load(classifier)
# TODO: Especify the args for the loading method
dataset_name = f"{bow_path}/validation"
image_classifier.predict(validation_set, dataset_name=dataset_name)



CLASSIFICATION RESULTS (../RESULTS/PARTA/BOW/VALIDATION)

Confusion matrix

KNOWN/PREDICTED  Bedroom  Coast  Forest  Highway  Industrial  Inside city  Kitchen  Living room  Mountain  Office  Open country  Store  Street  Suburb  Tall building
Bedroom             14.0    6.0     3.0      0.0         6.0         11.0      0.0         31.0       1.0     8.0           3.0    7.0     5.0     0.0            5.0
Coast                2.0   62.0     2.0      5.0         0.0          1.0      0.0          0.0      12.0     0.0          12.0    0.0     1.0     2.0            1.0
Forest               0.0    0.0    86.0      0.0         0.0          0.0      0.0          0.0       5.0     0.0           8.0    0.0     0.0     1.0            0.0
Highway              0.0   30.0     0.0     38.0         6.0          6.0      0.0          2.0       7.0     0.0           1.0    0.0     3.0     2.0            5.0
Industrial           4.0   10.0     0.0      0.0        35.0         19.0      0.0          

(0.4766666666666667,
 array([[14.,  6.,  3.,  0.,  6., 11.,  0., 31.,  1.,  8.,  3.,  7.,  5.,  0.,  5.],
        [ 2., 62.,  2.,  5.,  0.,  1.,  0.,  0., 12.,  0., 12.,  0.,  1.,  2.,  1.],
        [ 0.,  0., 86.,  0.,  0.,  0.,  0.,  0.,  5.,  0.,  8.,  0.,  0.,  1.,  0.],
        [ 0., 30.,  0., 38.,  6.,  6.,  0.,  2.,  7.,  0.,  1.,  0.,  3.,  2.,  5.],
        [ 4., 10.,  0.,  0., 35., 19.,  0.,  5.,  2.,  1.,  2.,  6., 11.,  1.,  4.],
        [ 1.,  0.,  0.,  1.,  7., 61.,  0.,  5.,  1.,  2.,  1.,  4.,  4.,  1., 12.],
        [ 9.,  0.,  0.,  0.,  3., 16., 10., 28.,  0., 15.,  1., 10.,  3.,  3.,  2.],
        [11.,  1.,  0.,  0., 10.,  4.,  1., 47.,  1.,  5.,  1.,  9.,  2.,  3.,  5.],
        [ 0., 12.,  4.,  0.,  0.,  0.,  0.,  0., 68.,  0.,  9.,  0.,  3.,  4.,  0.],
        [ 7.,  0.,  0.,  0.,  7., 23.,  5., 29.,  0., 20.,  0.,  3.,  2.,  2.,  2.],
        [ 0., 27.,  9.,  1.,  2.,  0.,  0.,  0., 16.,  0., 42.,  0.,  3.,  0.,  0.],
        [ 0.,  0.,  3.,  0., 11.,  8.,  0., 

### **Pregunta A.1**: Búsqueda de los mejores parámetros. ¿Cuántas palabras usar?

Entrene y evalúe el clasificador con vocabularios de 50, 100, 200 y 400 palabras (`vocabulary_size`). Elija además **uno** de estos parámetros y estudie su efecto: el kernel del SVM (`setKernel()` en `image_classifier.py`), el número de iteraciones del SVM (argumento `iterations` de `train()`) o el extractor de características (KAZE en lugar de SIFT). Muestre en una tabla la *accuracy* de entrenamiento y de validación de cada prueba y justifique qué configuración elegiría.

In [ ]:
#TODO: Homework